# 6장 비지도 학습: 군집과 차원 축소 — k-평균, 엘보·실루엣, DBSCAN, 주성분 분석, 이상 탐지
『머신러닝과 딥러닝 기초』 실습 노트북

다음을 해 봅니다.
1. k-평균의 세 단계를 넘파이로 직접 따라가기
2. 처음 중심과 스케일링에 따라 결과가 달라지는 것 확인하기
3. 엘보 방법과 실루엣 점수로 무리의 수 고르기, 특징을 늘렸을 때 달라지는 결과 보기
4. 반달 데이터에서 k-평균과 DBSCAN 비교하기
5. 주성분 분석(PCA)으로 펭귄 측정값과 손글씨 숫자 줄이기
6. 아이솔레이션 포레스트로 잘못 기록된 펭귄 찾기

위에서부터 차례로 실행하세요.

## 1. k-평균을 직접 따라가기
펭귄 데이터를 불러오고, 부리 길이와 부리 깊이 두 특징을 표준화합니다.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

num = ["bill_length_mm", "bill_depth_mm",
       "flipper_length_mm", "body_mass_g"]
df = sns.load_dataset("penguins").dropna(subset=num)
df = df.reset_index(drop=True)                     # 342마리
Z2 = StandardScaler().fit_transform(
    df[["bill_length_mm", "bill_depth_mm"]])  # 부리 길이, 부리 깊이

① 처음 중심을 무작위로 정하고, ② 가장 가까운 중심에 배정하고, ③ 무리의 평균으로 중심을 옮기는 일을 6번 되풀이합니다. 관성(자기 중심까지 거리 제곱의 합)이 어떻게 변하는지 보세요.

In [ ]:
rng = np.random.default_rng(1)
# 1. 처음 중심: 펭귄 3마리를 무작위로 고른다
centers = Z2[rng.choice(len(Z2), size=3, replace=False)]
for step in range(1, 7):
    # 2. 세 중심까지 거리의 제곱을 재고, 가장 가까운 중심에 배정
    d = np.column_stack(
        [((Z2 - c) ** 2).sum(axis=1) for c in centers])
    labels = d.argmin(axis=1)
    inertia = d.min(axis=1).sum()   # 자기 중심까지 거리 제곱의 합
    print(f"{step}회: 관성 {inertia:6.1f}, "
          f"무리 크기 {np.bincount(labels)}")
    # 3. 무리마다 속한 펭귄들의 평균으로 중심을 옮긴다
    centers = np.array(
        [Z2[labels == j].mean(axis=0) for j in range(3)])

사이킷런의 `KMeans`로 같은 일을 하면 관성과 무리 크기가 같게 나옵니다(무리 번호는 다를 수 있음).

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Z2)
print("관성:", round(km.inertia_, 1))
print("무리 크기:", np.bincount(km.labels_))

## 2. 처음 위치와 스케일링
`init="random"`(샘플 가운데 무작위로 처음 중심 고르기), `n_init=1`로 `random_state`를 0\~99로 바꾸어 100번 학습합니다. 관성이 가장 큰 결과는 실제 종과 맞대어 봅니다. 이북 6.2절에서는 결과(관성 186.9 대 303.4)만 그림으로 보였습니다.

참고: k-평균++는 첫 중심을 무작위로 고르고, 다음 중심부터는 이미 고른 중심 가운데 가장 가까운 것까지 거리의 제곱에 비례하는 확률로 샘플을 뽑습니다. 먼 샘플일수록 뽑히기 쉬워 처음 중심들이 퍼지지만, 나쁜 결과를 완전히 막지는 못합니다.

In [ ]:
res = pd.Series([KMeans(n_clusters=3, init="random", n_init=1,
                        random_state=s).fit(Z2).inertia_
                 for s in range(100)]).round(1)
print("관성: 나온 횟수", res.value_counts().to_dict())
print("관성이 가장 큰 random_state:", res.idxmax())
bad = KMeans(n_clusters=3, init="random", n_init=1,
             random_state=65).fit(Z2)
print(pd.crosstab(bad.labels_, df["species"],
                  rownames=["무리"], colnames=["실제 종"]))

`n_init=10`으로 여러 번 학습하면 어떻게 될까요? 기본값인 k-평균++로 처음 중심을 정하면 결과가 얼마나 달라지는지도 봅니다.

In [ ]:
km10 = KMeans(n_clusters=3, init="random", n_init=10,
              random_state=65).fit(Z2)
print(f"random, n_init=10: 관성 {km10.inertia_:.1f}")
for k in [3, 4]:   # 기본값 k-means++, n_init=1로 100번
    res = pd.Series([KMeans(n_clusters=k, n_init=1,
                            random_state=s).fit(Z2).inertia_
                     for s in range(100)])
    print(f"k-means++, k={k}: "
          f"관성 {res.min():.1f} ~ {res.max():.1f}")

$k=4$에서 100번 가운데 가장 작은 관성(147.7)은 이북 6.3절 출력의 $k=4$ 관성(147.8, `n_init=10`, `random_state=0`)보다 조금 작습니다. `n_init=10`으로 10번 해 보고 가장 좋은 것을 골라도 늘 가장 작은 관성을 찾는 것은 아닙니다.

스케일링하지 않고 묶으면 종과 얼마나 맞을까요? 무리마다 가장 많은 종의 수를 더해 비교합니다.

In [ ]:
def matched(labels):   # 무리마다 가장 많은 종의 수를 더한다(2장)
    ct = pd.crosstab(labels, df["species"])
    return ct.max(axis=1).sum()

X2 = df[["bill_length_mm", "bill_depth_mm"]]      # 표준화 전(mm)
raw = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X2)
print("스케일링 전:", matched(raw.labels_), "마리")
print("스케일링 후:", matched(km.labels_), "마리")

## 3. 몇 개로 묶을까: 엘보 방법과 실루엣 점수
$k$를 2부터 8까지 바꾸며 관성과 실루엣 점수를 구합니다.

In [ ]:
from sklearn.metrics import silhouette_score

for k in range(2, 9):
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z2)
    sil = silhouette_score(Z2, m.labels_)
    print(f"k={k}: 관성 {m.inertia_:6.1f}, 실루엣 {sil:.3f}")

이북 6.3절의 다섯 점 예(1, 2, 3 / 8, 9)를 `silhouette_samples`로 확인합니다. 점 3은 손으로 구한 0.727이 나와야 합니다.

In [ ]:
from sklearn.metrics import silhouette_samples

x = np.array([[1], [2], [3], [8], [9]])   # 수직선 위의 다섯 점
lab = [0, 0, 0, 1, 1]                      # A = {1, 2, 3}, B = {8, 9}
print("점마다:", silhouette_samples(x, lab).round(3))
print("평균:", round(silhouette_score(x, lab), 3))

$k=2$와 $k=3$의 결과를 실제 종과 맞대어 봅니다.

In [ ]:
for k in [2, 3]:
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z2)
    print(pd.crosstab(m.labels_, df["species"],
                      rownames=["무리"], colnames=["실제 종"]))

### 주의: 특징을 늘리면
측정값 네 개를 모두 표준화해 같은 일을 합니다(이북 6.3절 「주의」의 결과). 마지막 표는 아델리펭귄만 골라 무리와 성별을 맞댄 것입니다. 성별이 비어 있는 펭귄은 `fillna("모름")`으로 '모름'에 넣어 세므로, 행마다 더하면 위 표의 아델리 수(127, 24)와 같습니다.

In [ ]:
Z = StandardScaler().fit_transform(df[num])   # 측정값 네 개
for k in range(2, 7):
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    sil = silhouette_score(Z, m.labels_)
    print(f"k={k}: 관성 {m.inertia_:6.1f}, 실루엣 {sil:.3f}")
m3 = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Z)
print(pd.crosstab(m3.labels_, df["species"],
                  rownames=["무리"], colnames=["실제 종"]))
ad = (df["species"] == "Adelie").to_numpy()
print(pd.crosstab(m3.labels_[ad], df.loc[ad, "sex"].fillna("모름"),
                  rownames=["무리(아델리만)"], colnames=["성별"]))

## 4. k-평균이 못 하는 것: 반달 데이터와 DBSCAN

In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

Xm, ym = make_moons(n_samples=300, noise=0.07, random_state=0)
km_m = KMeans(n_clusters=2, n_init=10, random_state=0).fit(Xm)
db = DBSCAN(eps=0.2, min_samples=5).fit(Xm)
for name, m in [("k-평균", km_m), ("DBSCAN", db)]:
    print(name)
    print(pd.crosstab(m.labels_, ym, rownames=["무리"],
                      colnames=["실제 반달"]))

DBSCAN의 `eps`를 바꾸어 봅니다. 마지막 줄은 두 방법의 실루엣 점수입니다.

참고: 여기서는 `eps=0.2`의 잡음이 0개여서 실루엣 점수를 그대로 계산했습니다. DBSCAN 결과에 잡음(-1)이 있으면 `silhouette_score`는 잡음점들을 한 무리로 보고 계산하므로, 잡음을 뺀 샘플로만 계산해야 합니다.

In [ ]:
for eps in [0.05, 0.1, 0.2, 0.3]:
    lab = DBSCAN(eps=eps, min_samples=5).fit(Xm).labels_
    n_cl = len(set(lab) - {-1})       # 잡음(-1)을 뺀 무리 수
    print(f"eps={eps}: 무리 {n_cl}개, 잡음 {(lab == -1).sum()}개")
s_km = silhouette_score(Xm, km_m.labels_)
s_db = silhouette_score(Xm, db.labels_)
print(f"실루엣: k-평균 {s_km:.3f} / DBSCAN {s_db:.3f}")

## 5. 주성분 분석(PCA)
측정값 네 개(표준화)를 주성분 두 개로 줄이고, 설명된 분산 비율을 봅니다.

In [ ]:
from sklearn.decomposition import PCA

Z = StandardScaler().fit_transform(df[num])   # 측정값 네 개
pca = PCA(n_components=2)
P = pca.fit_transform(Z)          # 342×4 → 342×2
r = pca.explained_variance_ratio_
print("설명된 분산 비율:", r.round(3), "합:", r.sum().round(3))
pca_raw = PCA(n_components=2).fit(df[num])    # 표준화하지 않은 값
print("표준화하지 않으면:",
      pca_raw.explained_variance_ratio_.round(4))

주성분 두 개로 펭귄을 그려 봅니다(색은 실제 종, PCA는 종을 보지 않았습니다).

In [ ]:
import matplotlib.pyplot as plt

for sp, c in [("Adelie", "tab:orange"), ("Chinstrap", "tab:purple"), ("Gentoo", "tab:green")]:
    m = (df["species"] == sp).to_numpy()
    plt.scatter(P[m, 0], P[m, 1], s=15, color=c, label=sp)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend(); plt.show()

주성분은 원래 특징들의 조합입니다. 가중치(`components_`)를 읽어 봅니다.

In [ ]:
comp = pd.DataFrame(pca.components_.T, index=num,
                    columns=["PC1", "PC2"])
print(comp.round(3))
print("0번 펭귄(표준화):", Z[0].round(3))
print("0번 펭귄(주성분 점수):", P[0].round(3))

참고: 주성분은 직선이므로 가중치의 부호를 모두 바꿔도 같은 직선입니다. 그래서 사이킷런 버전이나 데이터에 따라 가중치와 점수의 부호가 통째로 뒤집혀 나올 수 있습니다(그림도 그 축으로 뒤집힙니다).

특징이 64개인 손글씨 숫자 데이터에서 PCA를 k-최근접 이웃의 준비 단계로 씁니다.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline

Xd, yd = load_digits(return_X_y=True)        # 1797장 × 64칸
Dtr, Dte, dtr, dte = train_test_split(
    Xd, yd, test_size=0.25, random_state=42, stratify=yd)
cum = PCA().fit(Dtr).explained_variance_ratio_.cumsum()
print("분산 90%를 넘기는 주성분 수:", (cum <= 0.9).sum() + 1)
for nc in [2, 21]:
    m = make_pipeline(PCA(n_components=nc),
                      KNeighborsClassifier())
    m.fit(Dtr, dtr)
    print(f"주성분 {nc:2d}개: 테스트 정확도 {m.score(Dte, dte):.1%}")
m = KNeighborsClassifier().fit(Dtr, dtr)
print(f"원래 특징 64개: 테스트 정확도 {m.score(Dte, dte):.1%}")

참고: `PCA(n_components=0.9)`처럼 0과 1 사이의 수를 주면, 설명된 분산 비율을 앞에서부터 더한 값이 처음으로 그 값을 넘을 때까지의 주성분을 남깁니다. 위에서 `cum`으로 구한 수와 같은지 확인합니다.

In [ ]:
p90 = PCA(n_components=0.9).fit(Dtr)   # 분산의 90%를 넘길 때까지 남긴다
print("n_components=0.9로 남은 주성분 수:", p90.n_components_)

주성분 두 개로 줄인 `P`로 다시 k-평균을 해 봅니다. 특징을 줄이면 군집이 좋아질까요?

In [ ]:
for k in range(2, 7):
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(P)
    print(f"k={k}: 실루엣 {silhouette_score(P, m.labels_):.3f}")
m3 = KMeans(n_clusters=3, n_init=10, random_state=0).fit(P)
print("k=3, 자기 종이 가장 많은 무리에 든 펭귄:", matched(m3.labels_))

### 심화: 공분산 행렬의 고윳값
이북 6.5절 심화 상자의 내용을 넘파이로 확인합니다. 먼저 표준화한 날개 길이와 몸무게의 $2 \times 2$ 공분산 행렬에 대각선 방향 화살표를 곱하면 길이만 $1+r$배가 되는지 봅니다. 이어서 측정값 네 개의 $4 \times 4$ 공분산 행렬에서 고윳값이 PCA가 구한 주성분마다의 분산(`explained_variance_`)과 같고, 고윳값을 모두 더한 값으로 나누면 설명된 분산 비율이 됩니다.

In [ ]:
W = StandardScaler().fit_transform(
    df[["flipper_length_mm", "body_mass_g"]])   # 날개 길이, 몸무게
r = np.corrcoef(W, rowvar=False)[0, 1]
C2 = np.array([[1, r], [r, 1]])     # 표준화한 두 특징의 공분산 행렬(n으로 나눔)
u = np.array([1, 1]) / np.sqrt(2)  # 대각선 방향
v = np.array([1, -1]) / np.sqrt(2) # 그와 직각인 방향
print("r:", round(r, 3))
print("C2 @ u:", (C2 @ u).round(3), "= (1 + r) * u:", ((1 + r) * u).round(3))
print("C2 @ v:", (C2 @ v).round(3), "= (1 - r) * v:", ((1 - r) * v).round(3))
print("2×2 고윳값:", np.linalg.eigh(C2)[0][::-1].round(3))

C = np.cov(Z, rowvar=False)        # 4×4 공분산 행렬
vals, vecs = np.linalg.eigh(C)     # 고윳값(작은 것부터), 고유벡터
ev = PCA().fit(Z).explained_variance_   # PCA가 구한 분산
print("4×4 고윳값:", vals[::-1].round(3))
print("PCA의 분산:", ev.round(3))
print("비율:", (vals[::-1] / vals.sum()).round(3))
print("고윳값의 합:", vals.sum().round(3))

고윳값의 합이 4가 아니라 4보다 조금 큰 약 4.012인 것은 `np.cov`와 PCA가 분산을 $n-1$로 나누어 구하고, `StandardScaler`는 $n$으로 나누어 표준화했기 때문입니다($4 \times 342/341 \approx 4.012$).

## 6. 이상 탐지: 잘못 기록된 펭귄 찾기
펭귄 데이터에 가짜 기록 5개(342\~346번)를 섞고, 아이솔레이션 포레스트로 찾아봅니다.

In [ ]:
from sklearn.ensemble import IsolationForest

fake = pd.DataFrame([[35.0, 14.0, 228, 3000],   # 가짜 기록 5개
                     [58.0, 21.0, 175, 6000],
                     [40.0, 20.5, 225, 6200],
                     [55.0, 14.0, 180, 3000],
                     [33.0, 20.0, 230, 5500]], columns=num)
out = (fake < df[num].min()) | (fake > df[num].max())
print("범위를 벗어난 값이 있는가:", out.any().any())
data = pd.concat([df[num], fake], ignore_index=True)   # 342 + 5
iso = IsolationForest(contamination=0.02, random_state=0)
flag = iso.fit(data).predict(data) == -1       # -1이면 이상
show = data[flag].set_axis(["bill_len", "bill_dep",
                            "flipper", "mass"], axis=1)
print(show.assign(score=iso.score_samples(data)[flag].round(3)))
auto = IsolationForest(random_state=0).fit(data)  # 기본값 사용
print("기본값으로 표시된 수:", (auto.predict(data) == -1).sum())

---
## 실습 문제

**문제 1.** 2장 2.6절에서 쓴 두 특징(부리 길이와 날개 길이)을 표준화해 k-평균으로 묶어 보세요. $k$를 2부터 6까지 바꾸어 관성과 실루엣 점수를 구하고, $k=3$의 결과를 실제 종과 맞대어 보세요. 6.3절의 부리 두 특징(부리 길이와 부리 깊이)과 비교하면 실루엣 점수가 가리키는 $k$와 종과 맞는 정도가 어떻게 다른가요?

In [ ]:
# 문제 1 풀이

**문제 2.** 측정값 네 개를 표준화한 `Z`에 DBSCAN(`min_samples=5`)을 적용해 보세요. `eps`를 0.3, 0.5, 0.7, 1.0으로 바꾸어 무리 수와 잡음 수를 구하고, 무리가 2개 이상이면 실제 종과 맞대어 보세요.

In [ ]:
# 문제 2 풀이

**문제 3.** 펭귄 종을 맞히는 k-최근접 이웃 모델에 PCA를 넣어 보세요. `make_pipeline(StandardScaler(), PCA(n_components=nc), KNeighborsClassifier())`에서 `nc`를 1\~4로 바꾸어 5-폴드 교차 검증 정확도를 비교하세요(특징은 측정값 네 개, 레이블은 종).

In [ ]:
# 문제 3 풀이

**문제 4.** 6절의 아이솔레이션 포레스트에서 `contamination`을 0.01, 0.02, 0.05로 바꾸어 보세요. 각각 몇 개가 표시되고 가짜 기록 5개 가운데 몇 개를 찾았나요? 4장의 정밀도와 재현율로 비교하세요.

In [ ]:
# 문제 4 풀이